In [1]:
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import time

In [2]:
X, y = make_classification(
    n_samples=1000, n_features=20,
    n_informative=5, n_redundant=10, random_state=42
)

In [3]:
X

array([[ 0.96923059, -4.85196274, -1.22401622, ...,  3.1499318 ,
         3.78843151,  2.05314018],
       [-0.41438979, -0.97201894,  0.9466201 , ...,  0.21325567,
         2.83013476,  0.15121834],
       [-1.96403216,  0.39379459,  2.31483331, ..., -1.14494475,
         0.54878615,  1.90112073],
       ...,
       [-0.77491653, -3.87987774,  2.44925379, ...,  1.13204737,
         2.91988478, -0.93286311],
       [ 0.06835044, -6.39745066,  2.02019502, ...,  2.59967376,
         2.68024151, -1.85747878],
       [ 0.13999374, -1.4246532 ,  0.79007549, ...,  0.50130311,
         1.73357455, -0.37656662]], shape=(1000, 20))

In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [5]:
selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X_scaled, y)

In [6]:
pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled)

In [7]:
scenarios = {
    '原始数据 (20D)': X_scaled,
    'SelectKBest (10D)': X_selected,
    'PCA (5D)': X_pca,
}

In [8]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=200, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
}

In [10]:
print("=" * 60)

for model_name, model in models.items():
    print(f"\n模型: {model_name}")
    print("-" * 60)
    print(f"{'数据方案':<22} {'维度':<6} {'准确率':<16} {'耗时(s)':<8}")
    print("-" * 60)

    for scenario_name, X_data in scenarios.items():
        start = time.time()

        scores = cross_val_score(model, X_data, y, cv=5, scoring='accuracy')
        elapsed = time.time() - start

        print(f"{scenario_name:<22} {X_data.shape[1]:<6} "
              f"{scores.mean():.4f} ± {scores.std():.4f}  "
              f"{elapsed:.3f}")

    print("=" * 60)


模型: LogisticRegression
------------------------------------------------------------
数据方案                   维度     准确率              耗时(s)   
------------------------------------------------------------
原始数据 (20D)             20     0.8220 ± 0.0242  0.028
SelectKBest (10D)      10     0.8220 ± 0.0254  0.017
PCA (5D)               5      0.8180 ± 0.0196  0.014

模型: RandomForest
------------------------------------------------------------
数据方案                   维度     准确率              耗时(s)   
------------------------------------------------------------
原始数据 (20D)             20     0.9470 ± 0.0191  0.921
SelectKBest (10D)      10     0.9380 ± 0.0191  0.737
PCA (5D)               5      0.9280 ± 0.0093  0.638
